# Orbit Wars - cnn_v1 Self-Play PPO (v16, T4 only)

Reverted to Kaggle's preinstalled torch 2.10+cu128. **Requires T4 (SM_75) or newer** —
P100 (SM_60) is not supported by cu128 kernels. Set **Settings → Accelerator → GPU T4 x2**
before running, otherwise the CUDA smoke test will fail fast.


In [ ]:
!pip install -q --upgrade 'kaggle-environments>=1.28.0'


## Detect runtime + set paths

In [ ]:
import os, sys, shutil
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
IS_KAGGLE = Path('/kaggle/working').exists()

if IS_COLAB:
    BASE = Path('/content/orbit-wars')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CKPT_DIR = Path('/content/drive/MyDrive/orbit-wars/checkpoints')
    except Exception as e:
        print('Drive mount failed:', e)
        CKPT_DIR = BASE / 'checkpoints'
elif IS_KAGGLE:
    BASE = Path('/kaggle/working/orbit-wars')
    CKPT_DIR = Path('/kaggle/working/checkpoints')
    kaggle_inputs = Path('/kaggle/input')
    if kaggle_inputs.exists():
        for src in kaggle_inputs.iterdir():
            prev = src / 'checkpoints'
            if prev.exists():
                CKPT_DIR.mkdir(parents=True, exist_ok=True)
                for f in prev.glob('*.pt'):
                    shutil.copy(f, CKPT_DIR / f.name)
                print(f'seeded CKPT_DIR from {prev}:', list(CKPT_DIR.iterdir()))
                break
else:
    BASE = Path('./orbit-wars').resolve()
    CKPT_DIR = Path('./checkpoints').resolve()

BASE.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(BASE)

print(f'env: {"colab" if IS_COLAB else "kaggle" if IS_KAGGLE else "local"}')
print(f'BASE     = {BASE}')
print(f'CKPT_DIR = {CKPT_DIR}')

import kaggle_environments
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet
print('kaggle_environments:', kaggle_environments.__version__, 'orbit_wars: OK')

import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cc = torch.cuda.get_device_capability(0)
    print(f'gpu: {name}  cc: {cc}')
    if cc < (7, 0):
        raise RuntimeError(
            f'Assigned GPU ({name}, SM_{cc[0]}{cc[1]}) is not supported by torch cu128. '
            f'Change Settings → Accelerator → GPU T4 x2, then Save & Run All.'
        )
    # Kernel smoke test — fails fast if architecture is incompatible.
    (torch.zeros(16, device='cuda') + 1.0).cpu()
    print('cuda kernel test: OK')


## Source: registry

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable

AgentFn = Callable[[dict], list]


@dataclass(frozen=True)
class AgentSpec:
    id: str
    fn: AgentFn
    description: str


_REGISTRY: dict[str, AgentSpec] = {}


def register(id: str, description: str):
    def decorator(fn: AgentFn) -> AgentFn:
        if id in _REGISTRY:
            raise ValueError(f"agent id {id!r} already registered")
        _REGISTRY[id] = AgentSpec(id=id, fn=fn, description=description)
        return fn

    return decorator


def list_agents() -> list[str]:
    return sorted(_REGISTRY)


def list_agent_specs() -> list[AgentSpec]:
    return [_REGISTRY[k] for k in sorted(_REGISTRY)]


class Agent:
    def __init__(self, id: str):
        if id not in _REGISTRY:
            raise KeyError(f"unknown agent id {id!r}. available: {list_agents()}")
        spec = _REGISTRY[id]
        self.id = spec.id
        self.fn: AgentFn = spec.fn
        self.description = spec.description

    def __call__(self, obs):
        return self.fn(obs)

    def __repr__(self) -> str:
        return f"Agent(id={self.id!r})"


## Source: utils/runner

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any


_REPO_ROOT = Path.cwd()
REPLAY_ROOT = _REPO_ROOT / "logs" / "replays"


def make_run_id(mode: str, agent_ids: list[str]) -> str:
    """Return a run id like ``play-20260421T013045-sniper-vs-random``."""
    ts = datetime.now().strftime("%Y%m%dT%H%M%S")
    combo = "-vs-".join(agent_ids)
    return f"{mode}-{ts}-{combo}"


def compute_scores(env) -> list[list[int]]:
    """Per-step total ship count (planets + fleets) per player."""
    scores: list[list[int]] = []
    for step in env.steps:
        if not step:
            continue
        obs = step[0].observation
        planets = obs.get("planets") or []
        fleets = obs.get("fleets") or []
        n = len(step)
        per_player = [0] * n
        for p in planets:
            owner, ships = p[1], p[5]
            if 0 <= owner < n:
                per_player[owner] += ships
        for f in fleets:
            owner, ships = f[1], f[6]
            if 0 <= owner < n:
                per_player[owner] += ships
        scores.append(per_player)
    return scores


@dataclass
class MatchResult:
    agent_ids: list[str]
    env: Any
    scores: list[list[int]]
    rewards: list
    winner: int


def _validate_and_load(agent_ids: list[str]) -> list[Agent]:
    if len(agent_ids) not in (2, 4):
        raise ValueError(f"orbit_wars requires 2 or 4 agents, got {len(agent_ids)}")
    return [Agent(id=a) for a in agent_ids]


def run_match(
    agent_ids: list[str],
    seed: int | None = None,
    debug: bool = False,
) -> MatchResult:
    """Play one match and return env + scores + rewards + winner."""
    from kaggle_environments import make

    players = _validate_and_load(agent_ids)
    config: dict = {"seed": seed} if seed is not None else {}
    env = make("orbit_wars", configuration=config, debug=debug)
    env.run([p.fn for p in players])

    scores = compute_scores(env)
    final = env.steps[-1]
    rewards = [s.reward for s in final]
    ranked = [r if r is not None else float("-inf") for r in rewards]
    winner = max(range(len(ranked)), key=lambda i: ranked[i])

    return MatchResult(
        agent_ids=agent_ids,
        env=env,
        scores=scores,
        rewards=rewards,
        winner=winner,
    )


def save_replay(env, path: Path | str) -> Path:
    """Write ``env.render(mode='html')`` to ``path`` and return it."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(env.render(mode="html"))
    return path


def train_match(*args, **kwargs):
    """Placeholder — self-play training loop (not implemented)."""
    raise NotImplementedError("training mode is not implemented yet")


## Source: physical_v2 agent

In [ ]:
"""Heuristic physical agent v2 — v1 + defensive threat accounting.

Single improvement over v1: before picking a target from a source planet,
compute the minimum garrison the planet will see over the timeline of
incoming enemy fleets. Only the *surplus* above that minimum (minus a
defensive buffer) is available for offense. Sources under threat hold.

Threat model per source:
  1. For each enemy fleet in flight, compute when its straight-line
     trajectory comes within the planet's radius (None if it misses or
     is moving away).
  2. Sort threats by arrival turn; walk the timeline forward:
       garrison(t) = garrison(last_t) + production · (t − last_t) − threat_ships(t)
     Track the minimum garrison across all arrivals.
  3. Surplus = min_garrison − DEFENSE_BUFFER.

Everything else (lead-aim, sun-dodge, rotation sign inference,
travel_time/production scoring, production-during-travel allocation) is
copied verbatim from v1 so the file stays self-contained for packing.
"""

from __future__ import annotations

import math

from kaggle_environments.envs.orbit_wars.orbit_wars import Fleet, Planet


SUN_CX = 50.0
SUN_CY = 50.0
SUN_RADIUS = 10.0
SUN_MARGIN = 1.0
MAX_SPEED = 6.0
SPEED_LOG_DENOM = math.log(1000.0)
ROTATION_RADIUS_LIMIT = 50.0
LEAD_AIM_ITERS = 6
SAFETY_BUFFER = 3
MIN_LAUNCH_SHIPS = 5
NEUTRAL_BONUS = 0.8
DEFENSE_BUFFER = 3


def fleet_speed(ships: int) -> float:
    if ships <= 1:
        return 1.0
    return 1.0 + (MAX_SPEED - 1.0) * (math.log(ships) / SPEED_LOG_DENOM) ** 1.5


def _dist_from_sun(x: float, y: float) -> float:
    return math.hypot(x - SUN_CX, y - SUN_CY)


def is_orbiting(p: Planet, angular_velocity: float) -> bool:
    if angular_velocity == 0:
        return False
    return (_dist_from_sun(p.x, p.y) + p.radius) < ROTATION_RADIUS_LIMIT


def crosses_sun(x1: float, y1: float, x2: float, y2: float) -> bool:
    dx, dy = x2 - x1, y2 - y1
    len_sq = dx * dx + dy * dy
    if len_sq == 0:
        return _dist_from_sun(x1, y1) <= SUN_RADIUS + SUN_MARGIN
    t = max(0.0, min(1.0, ((SUN_CX - x1) * dx + (SUN_CY - y1) * dy) / len_sq))
    cx, cy = x1 + t * dx, y1 + t * dy
    return math.hypot(cx - SUN_CX, cy - SUN_CY) <= SUN_RADIUS + SUN_MARGIN


def infer_rotation_sign(planets: list[Planet], initial_planets: list) -> int:
    init = {row[0]: row for row in initial_planets}
    for p in planets:
        if p.id not in init:
            continue
        ip = init[p.id]
        ix, iy = ip[2], ip[3]
        ir = math.hypot(ix - SUN_CX, iy - SUN_CY)
        cr = _dist_from_sun(p.x, p.y)
        if abs(ir - cr) > 0.5:
            continue
        ia = math.atan2(iy - SUN_CY, ix - SUN_CX)
        ca = math.atan2(p.y - SUN_CY, p.x - SUN_CX)
        delta = (ca - ia + math.pi) % (2 * math.pi) - math.pi
        if abs(delta) > 1e-3:
            return 1 if delta > 0 else -1
    return 1


def predict_position(p: Planet, av_signed: float, turns: float) -> tuple[float, float]:
    dx, dy = p.x - SUN_CX, p.y - SUN_CY
    r = math.hypot(dx, dy)
    angle = math.atan2(dy, dx) + av_signed * turns
    return SUN_CX + r * math.cos(angle), SUN_CY + r * math.sin(angle)


def lead_aim(
    source: Planet,
    target: Planet,
    fleet_ships: int,
    av_signed: float,
    orbiting: bool,
) -> tuple[float, float, float]:
    speed = fleet_speed(fleet_ships)
    if not orbiting:
        dist = math.hypot(target.x - source.x, target.y - source.y)
        return target.x, target.y, dist / speed
    px, py = target.x, target.y
    turns = math.hypot(px - source.x, py - source.y) / speed
    for _ in range(LEAD_AIM_ITERS):
        px, py = predict_position(target, av_signed, turns)
        turns = math.hypot(px - source.x, py - source.y) / speed
    return px, py, turns


def fleet_eta_to_planet(fleet: Fleet, planet: Planet) -> float | None:
    """Time until ``fleet``'s trajectory comes within ``planet.radius``.

    Returns None if the fleet misses the planet or is moving away.
    Treats the planet as static — good enough for a v2 threat heuristic;
    orbiting planets that move away can only over-count threat, which
    the DEFENSE_BUFFER already absorbs.
    """
    speed = fleet_speed(fleet.ships)
    if speed <= 0:
        return None
    ch = math.cos(fleet.angle)
    sh = math.sin(fleet.angle)
    dx = planet.x - fleet.x
    dy = planet.y - fleet.y
    t_closest = (dx * ch + dy * sh) / speed
    if t_closest < 0:
        return None
    cx = fleet.x + speed * t_closest * ch
    cy = fleet.y + speed * t_closest * sh
    if math.hypot(cx - planet.x, cy - planet.y) > planet.radius:
        return None
    return t_closest


def compute_surplus(source: Planet, enemy_fleets: list[Fleet]) -> int:
    """Min garrison over the threat timeline, minus DEFENSE_BUFFER."""
    threats = []
    for f in enemy_fleets:
        eta = fleet_eta_to_planet(f, source)
        if eta is not None:
            threats.append((eta, f.ships))
    if not threats:
        return max(0, source.ships - DEFENSE_BUFFER)
    threats.sort(key=lambda x: x[0])
    garrison = float(source.ships)
    min_garrison = garrison
    last_t = 0.0
    for t, ships in threats:
        garrison += source.production * (t - last_t)
        garrison -= ships
        if garrison < min_garrison:
            min_garrison = garrison
        last_t = t
    return int(max(0, math.floor(min_garrison) - DEFENSE_BUFFER))


def _score(turns: float, target: Planet) -> float:
    base = turns / max(1, target.production)
    return base * (NEUTRAL_BONUS if target.owner == -1 else 1.0)


@register(
    "physical_v2",
    "physical_v1 + defensive threat accounting. Incoming enemy fleet trajectories "
    "constrain the launch budget per planet so threatened sources hold.",
)
def physical_v2_agent(obs):
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    get = obs.get if isinstance(obs, dict) else lambda k, d=None: getattr(obs, k, d)
    raw_planets = get("planets") or []
    raw_fleets = get("fleets") or []
    angular_velocity = abs(float(get("angular_velocity") or 0.0))
    initial_planets = get("initial_planets") or []

    planets = [Planet(*p) for p in raw_planets]
    fleets = [Fleet(*f) for f in raw_fleets]
    av_sign = infer_rotation_sign(planets, initial_planets)
    av_signed = angular_velocity * av_sign

    my_planets = [p for p in planets if p.owner == player and p.ships >= MIN_LAUNCH_SHIPS]
    targets = [p for p in planets if p.owner != player]
    enemy_fleets = [f for f in fleets if f.owner != player and f.owner >= 0]
    if not my_planets or not targets:
        return []

    moves = []
    for source in my_planets:
        surplus = compute_surplus(source, enemy_fleets)
        if surplus < MIN_LAUNCH_SHIPS:
            continue

        best = None
        best_score = float("inf")
        best_angle = 0.0
        best_ships = 0

        for target in targets:
            orbiting = is_orbiting(target, angular_velocity)
            fleet_guess = min(max(target.ships + 10, 20), surplus)
            if fleet_guess < 1:
                continue

            px, py, turns = lead_aim(source, target, fleet_guess, av_signed, orbiting)
            if crosses_sun(source.x, source.y, px, py):
                continue

            if target.owner == -1:
                ships_on_arrival = target.ships
            else:
                ships_on_arrival = target.ships + int(target.production * turns)
            ships_needed = ships_on_arrival + SAFETY_BUFFER

            if ships_needed > surplus:
                continue

            sc = _score(turns, target)
            if sc < best_score:
                best_score = sc
                best = target
                best_angle = math.atan2(py - source.y, px - source.x)
                best_ships = ships_needed

        if best is not None and best_ships > 0:
            moves.append([source.id, best_angle, best_ships])

    return moves


## Source: cnn_v1 agent

In [ ]:
"""CNN v1 agent — convolutional policy over a player-relative 50×50 raster.

--------------------------------------------------------------------------
Input channels (shape: 15 × 50 × 50, cell = 2 world units)
--------------------------------------------------------------------------
| idx | channel                    | what it encodes                              |
|-----|----------------------------|----------------------------------------------|
|  0  | my_planet_ships            | log1p(ships)/log(5000) at my planets         |
|  1  | enemy_planet_ships         | ... at enemy planets                         |
|  2  | neutral_planet_ships       | ... at neutral planets                       |
|  3  | my_planet_production       | production/5 at my planets                   |
|  4  | enemy_planet_production    | ... at enemy planets                         |
|  5  | neutral_planet_production  | ... at neutral planets                       |
|  6  | my_fleet_ships             | log1p(ships)/log(5000) at my fleets          |
|  7  | enemy_fleet_ships          | ... at enemy fleets                          |
|  8  | my_fleet_sin               | sin(heading) at my fleets                    |
|  9  | my_fleet_cos               | cos(heading) at my fleets                    |
| 10  | enemy_fleet_sin            | sin(heading) at enemy fleets                 |
| 11  | enemy_fleet_cos            | cos(heading) at enemy fleets                 |
| 12  | orbit_mask                 | 1 on cells containing an orbiting planet     |
| 13  | comet_mask                 | 1 on cells containing a comet                |
| 14  | sun_mask                   | static disc at grid cell (25, 25), radius 5  |

Scalar side-channel (7 floats, concatenated after the backbone):
  step/500, log1p(my_ships)/log5k, log1p(enemy_ships)/log5k,
  my_planet_count/10, enemy_planet_count/10, angular_velocity,
  active_comet_count/20.

--------------------------------------------------------------------------
Backbone (receptive field ≈ 33 cells ≈ 66 world units)
--------------------------------------------------------------------------
  Conv(15 → 32, 3×3)                               ReLU  RF=3
  Conv(32 → 64, 3×3)                               ReLU  RF=5
  Conv(64 → 64, 3×3, dilation=2)                   ReLU  RF=9
  Conv(64 → 64, 3×3, dilation=4)                   ReLU  RF=17
  Conv(64 → 64, 3×3, dilation=8)                   ReLU  RF=33

Scalar MLP: 7 → 32 → 32 (ReLU).

--------------------------------------------------------------------------
Action head (per owned planet)
--------------------------------------------------------------------------
  bilinear-sample feature map at planet (x, y) → concat with scalar MLP
  output → MLP(96 → 96) → three heads:

    launch_logit      ∈ ℝ           (sigmoid → launch gate)
    target_logits     ∈ ℝ^(50×50)   (argmax → target cell)
    ship_fraction_lg  ∈ ℝ           (sigmoid → ship fraction of garrison)

The target cell → angle via atan2(ty − py, tx − px); ships =
round(fraction × garrison), clipped to [1, garrison].

Status: untrained. Default weights → near-random valid actions. When
WEIGHTS_PATH exists, it loads on first call.
"""

from __future__ import annotations

import math
import warnings
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


BOARD_SIZE = 100.0
GRID = 50
CELL = BOARD_SIZE / GRID
NUM_CHANNELS = 15
SCALAR_DIM = 7
D_MODEL = 64
MAX_STEPS = 500.0
SHIPS_LOG_MAX = math.log(5000.0)
LAUNCH_THRESHOLD = 0.5

WEIGHTS_PATH = Path.cwd() / "cnn_v1.pt"


def _cell(v: float) -> int:
    return max(0, min(GRID - 1, int(v / CELL)))


def featurize(obs):
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    get = obs.get if isinstance(obs, dict) else lambda k, d=None: getattr(obs, k, d)
    raw_planets = get("planets") or []
    raw_fleets = get("fleets") or []
    comet_ids = set(get("comet_planet_ids") or [])
    angular_velocity = float(get("angular_velocity") or 0.0)
    step = int(get("step", 0) or 0)

    ch = torch.zeros(NUM_CHANNELS, GRID, GRID, dtype=torch.float32)

    sy, sx = torch.meshgrid(torch.arange(GRID), torch.arange(GRID), indexing="ij")
    dist = ((sx - GRID / 2) ** 2 + (sy - GRID / 2) ** 2).sqrt()
    ch[14] = (dist <= 5).float()

    my_ships = 0
    enemy_ships = 0
    my_planet_count = 0
    enemy_planet_count = 0
    my_planets = []

    for p in raw_planets:
        pid, owner, x, y, radius, ships, prod = p
        cx, cy = _cell(x), _cell(y)
        log_ships = math.log1p(ships) / SHIPS_LOG_MAX
        prod_norm = prod / 5.0
        is_orbiting = angular_velocity > 0 and math.hypot(x - 50, y - 50) + radius < 50
        if owner == player:
            ch[0, cy, cx] += log_ships
            ch[3, cy, cx] += prod_norm
            my_ships += ships
            my_planet_count += 1
            my_planets.append((pid, x, y, ships))
        elif owner == -1:
            ch[2, cy, cx] += log_ships
            ch[5, cy, cx] += prod_norm
        else:
            ch[1, cy, cx] += log_ships
            ch[4, cy, cx] += prod_norm
            enemy_ships += ships
            enemy_planet_count += 1
        if is_orbiting:
            ch[12, cy, cx] = 1.0
        if pid in comet_ids:
            ch[13, cy, cx] = 1.0

    for f in raw_fleets:
        fid, owner, x, y, angle, from_pid, ships = f
        cx, cy = _cell(x), _cell(y)
        log_ships = math.log1p(ships) / SHIPS_LOG_MAX
        if owner == player:
            ch[6, cy, cx] += log_ships
            ch[8, cy, cx] = math.sin(angle)
            ch[9, cy, cx] = math.cos(angle)
            my_ships += ships
        elif owner != -1:
            ch[7, cy, cx] += log_ships
            ch[10, cy, cx] = math.sin(angle)
            ch[11, cy, cx] = math.cos(angle)
            enemy_ships += ships

    scalars = torch.tensor(
        [
            step / MAX_STEPS,
            math.log1p(my_ships) / SHIPS_LOG_MAX,
            math.log1p(enemy_ships) / SHIPS_LOG_MAX,
            my_planet_count / 10.0,
            enemy_planet_count / 10.0,
            angular_velocity,
            len(comet_ids) / 20.0,
        ],
        dtype=torch.float32,
    )
    return ch, scalars, my_planets


class CNNv1(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(NUM_CHANNELS, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, D_MODEL, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(D_MODEL, D_MODEL, 3, padding=2, dilation=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(D_MODEL, D_MODEL, 3, padding=4, dilation=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(D_MODEL, D_MODEL, 3, padding=8, dilation=8),
            nn.ReLU(inplace=True),
        )
        self.scalar_mlp = nn.Sequential(
            nn.Linear(SCALAR_DIM, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 32),
            nn.ReLU(inplace=True),
        )
        self.head_trunk = nn.Sequential(
            nn.Linear(D_MODEL + 32, 96),
            nn.ReLU(inplace=True),
        )
        self.launch_head = nn.Linear(96, 1)
        self.target_head = nn.Linear(96, GRID * GRID)
        self.ship_head = nn.Linear(96, 1)
        self.value_head = nn.Sequential(
            nn.Linear(D_MODEL + 32, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 1),
        )
        self.frac_log_std = nn.Parameter(torch.log(torch.tensor(0.2)))

    def forward(self, channels, scalars):
        feat_map = self.conv(channels)
        scalar_feat = self.scalar_mlp(scalars)
        return feat_map, scalar_feat

    def value(self, feat_map, scalar_feat):
        pooled = feat_map.mean(dim=(2, 3))
        return self.value_head(torch.cat([pooled, scalar_feat], dim=-1)).squeeze(-1)

    def act(self, feat_map, scalar_feat, planet_coords):
        n = planet_coords.size(0)
        if n == 0:
            empty = torch.empty(0, dtype=torch.float32)
            return empty, torch.empty(0, GRID * GRID), empty

        grid = planet_coords.clone()
        grid[:, 0] = (grid[:, 0] / BOARD_SIZE) * 2 - 1
        grid[:, 1] = (grid[:, 1] / BOARD_SIZE) * 2 - 1
        grid = grid.view(1, n, 1, 2)
        sampled = F.grid_sample(feat_map, grid, mode="bilinear", align_corners=False)
        per_planet = sampled.squeeze(-1).squeeze(0).transpose(0, 1)

        s = scalar_feat.expand(n, -1)
        x = self.head_trunk(torch.cat([per_planet, s], dim=-1))
        return (
            self.launch_head(x).squeeze(-1),
            self.target_head(x),
            self.ship_head(x).squeeze(-1),
        )


_MODEL: CNNv1 | None = None


def _get_model() -> CNNv1:
    global _MODEL
    if _MODEL is None:
        m = CNNv1()
        m.eval()
        if WEIGHTS_PATH.exists():
            try:
                m.load_state_dict(torch.load(WEIGHTS_PATH, map_location="cpu"), strict=False)
            except Exception as e:
                warnings.warn(f"cnn_v1: failed to load weights: {e}")
        _MODEL = m
    return _MODEL


def reload_weights() -> None:
    """Invalidate the cached model so the next call reloads from disk."""
    global _MODEL
    _MODEL = None


@register(
    "cnn_v1",
    "Convolutional policy over a 50×50 player-relative raster — "
    "15 channels, dilated stack, per-planet action head. Untrained.",
)
def cnn_v1_agent(obs):
    channels, scalars, my_planets = featurize(obs)
    if not my_planets:
        return []

    model = _get_model()
    with torch.no_grad():
        feat_map, scalar_feat = model(channels.unsqueeze(0), scalars.unsqueeze(0))
        coords = torch.tensor([[x, y] for (_, x, y, _) in my_planets], dtype=torch.float32)
        launch, targets, ship_frac = model.act(feat_map, scalar_feat, coords)

    launch_p = torch.sigmoid(launch)
    ship_p = torch.sigmoid(ship_frac)
    target_cells = targets.argmax(dim=-1)

    moves = []
    for i, (pid, x, y, ships) in enumerate(my_planets):
        if launch_p[i].item() < LAUNCH_THRESHOLD:
            continue
        tc = int(target_cells[i].item())
        tcy, tcx = divmod(tc, GRID)
        tx = (tcx + 0.5) * CELL
        ty = (tcy + 0.5) * CELL
        n = max(1, min(int(ships), int(round(ship_p[i].item() * ships))))
        if n < 1:
            continue
        angle = math.atan2(ty - y, tx - x)
        moves.append([pid, angle, n])

    return moves


## Source: cnn_v1 common

In [ ]:
"""Shared helpers for the CNN v1 training pipeline.

Bridges the env's action format (`[[from_planet_id, angle, num_ships], ...]`)
and the policy's per-planet output tuples (launch, target_cell, ship_frac).
"""

from __future__ import annotations

import math
from pathlib import Path


RAY_DIST = 30.0  # units along the angle that we treat as the "target" of a launch

_REPO_ROOT = Path.cwd()
TRAIN_ROOT = _REPO_ROOT / "logs" / "training"
WEIGHTS_DIR = Path(WEIGHTS_PATH).parent


def angle_to_cell(px: float, py: float, angle: float, dist: float = RAY_DIST) -> int:
    """Project a ray from (px, py) at `angle` distance `dist`, snap to a grid cell."""
    tx = max(0.0, min(BOARD_SIZE - 0.001, px + dist * math.cos(angle)))
    ty = max(0.0, min(BOARD_SIZE - 0.001, py + dist * math.sin(angle)))
    cx = min(GRID - 1, int(tx / CELL))
    cy = min(GRID - 1, int(ty / CELL))
    return cy * GRID + cx


def decode_teacher_action(moves, my_planets):
    """Turn `[[from_id, angle, ships], ...]` into per-planet labels.

    Returns three lists aligned with `my_planets`:
      launched[i]     : 1.0 if planet i had any launches, else 0.0
      target_cell[i]  : int in [0, GRID*GRID) — first target cell if launched, else 0
      ship_frac[i]    : sum_of_ships / garrison, clipped to (0, 1], else 0

    For planets with multiple launches in a turn we use the first launch's
    target and sum the ships — good enough for a behavior-cloning warm-start.
    """
    id_to_idx = {p[0]: i for i, p in enumerate(my_planets)}
    n = len(my_planets)
    launched = [0.0] * n
    target_cell = [0] * n
    ship_frac = [0.0] * n
    ship_sum = [0] * n

    for m in moves:
        if not isinstance(m, (list, tuple)) or len(m) < 3:
            continue
        pid, angle, ships = m[0], float(m[1]), int(m[2])
        if pid not in id_to_idx:
            continue
        i = id_to_idx[pid]
        _, x, y, garrison = my_planets[i]
        if launched[i] == 0.0:
            target_cell[i] = angle_to_cell(x, y, angle)
        launched[i] = 1.0
        ship_sum[i] += ships
        ship_frac[i] = min(1.0, ship_sum[i] / max(1, garrison))
    return launched, target_cell, ship_frac


def fresh_model() -> CNNv1:
    return CNNv1()


def save_model(model: CNNv1, path: Path | str | None = None) -> Path:
    import torch

    target = Path(path) if path else WEIGHTS_PATH
    target.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), target)
    return target


def load_model(path: Path | str | None = None) -> CNNv1:
    import torch

    src = Path(path) if path else WEIGHTS_PATH
    m = fresh_model()
    if src.exists():
        m.load_state_dict(torch.load(src, map_location="cpu"), strict=False)
    return m


## Source: cnn_v1 bc helpers

In [ ]:
"""Behavior cloning trainer — warm-start the CNN on a teacher policy."""

from __future__ import annotations

import sys
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset



def _planet_features(model: CNNv1, channels: torch.Tensor, scalars: torch.Tensor, coords: torch.Tensor):
    """Mirror of CNNv1.act but batched. coords is (B, M, 2)."""
    feat_map = model.conv(channels)
    scalar_feat = model.scalar_mlp(scalars)
    B, M, _ = coords.shape
    grid = coords.clone()
    grid[..., 0] = (grid[..., 0] / 100.0) * 2 - 1
    grid[..., 1] = (grid[..., 1] / 100.0) * 2 - 1
    grid = grid.view(B, M, 1, 2)
    sampled = F.grid_sample(feat_map, grid, mode="bilinear", align_corners=False)  # (B, D, M, 1)
    per_planet = sampled.squeeze(-1).permute(0, 2, 1)  # (B, M, D)
    sfeat = scalar_feat.unsqueeze(1).expand(-1, M, -1)  # (B, M, 32)
    x = model.head_trunk(torch.cat([per_planet, sfeat], dim=-1))  # (B, M, 96)
    launch = model.launch_head(x).squeeze(-1)  # (B, M)
    target = model.target_head(x)  # (B, M, GRID*GRID)
    ship = model.ship_head(x).squeeze(-1)  # (B, M)
    return launch, target, ship


def train_bc(
    dataset: dict[str, torch.Tensor],
    epochs: int = 3,
    batch_size: int = 32,
    lr: float = 1e-3,
    launch_weight: float = 1.0,
    target_weight: float = 1.0,
    frac_weight: float = 0.5,
    resume_from: Path | str | None = None,
    save_to: Path | str | None = None,
    verbose: bool = True,
) -> CNNv1:
    model = load_model(resume_from) if resume_from else fresh_model()
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    ds = TensorDataset(
        dataset["channels"],
        dataset["scalars"],
        dataset["coords"],
        dataset["launched"],
        dataset["targets"],
        dataset["fracs"],
        dataset["mask"],
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0)

    for epoch in range(epochs):
        t0 = time.time()
        sums = {"launch": 0.0, "target": 0.0, "frac": 0.0, "total": 0.0, "n": 0}
        for batch in loader:
            channels, scalars, coords, launched, targets, fracs, mask = batch
            pred_launch, pred_target, pred_frac = _planet_features(model, channels, scalars, coords)

            # Mask to live planets
            m = mask.bool()
            if not m.any():
                continue

            launch_loss = F.binary_cross_entropy_with_logits(
                pred_launch[m], launched[m], reduction="mean"
            )

            lm = m & launched.bool()
            if lm.any():
                target_loss = F.cross_entropy(pred_target[lm], targets[lm], reduction="mean")
                frac_loss = F.mse_loss(torch.sigmoid(pred_frac[lm]), fracs[lm], reduction="mean")
            else:
                target_loss = torch.zeros((), device=pred_target.device)
                frac_loss = torch.zeros((), device=pred_frac.device)

            loss = (
                launch_weight * launch_loss
                + target_weight * target_loss
                + frac_weight * frac_loss
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            sums["launch"] += launch_loss.item()
            sums["target"] += target_loss.item()
            sums["frac"] += frac_loss.item()
            sums["total"] += loss.item()
            sums["n"] += 1

        if verbose and sums["n"]:
            n = sums["n"]
            print(
                f"epoch {epoch + 1}/{epochs}  "
                f"launch={sums['launch']/n:.4f}  "
                f"target={sums['target']/n:.4f}  "
                f"frac={sums['frac']/n:.4f}  "
                f"total={sums['total']/n:.4f}  "
                f"({time.time() - t0:.1f}s)",
                file=sys.stderr,
            )

    model.eval()
    if save_to is not None or save_to is None:
        path = save_model(model, save_to)
        if verbose:
            print(f"saved weights: {path}", file=sys.stderr)
    return model


## Source: cnn_v1 eval

In [ ]:
"""Tournament evaluator — play N games against each opponent, report win rate."""

from __future__ import annotations

import sys
from typing import Iterable


DEFAULT_OPPONENTS = ["random_v1", "sniper_v1", "physical_v2"]


def evaluate_agent(
    agent_id: str,
    opponents: Iterable[str] = DEFAULT_OPPONENTS,
    games_per: int = 10,
    seed_start: int = 0,
    verbose: bool = True,
) -> dict[str, dict[str, int]]:
    """Play `games_per` games vs each opponent, alternating seats.

    Returns: ``{opponent: {"wins": w, "losses": l, "draws": d, "win_rate": p}}``.
    """
    results: dict[str, dict[str, int]] = {}
    for opp in opponents:
        wins = losses = draws = 0
        for g in range(games_per):
            agent_slot = g % 2  # alternate who is P0
            ids = [agent_id, opp] if agent_slot == 0 else [opp, agent_id]
            r = run_match(ids, seed=seed_start + g)
            if r.winner == agent_slot:
                wins += 1
            elif r.winner == 1 - agent_slot:
                losses += 1
            else:
                draws += 1
        total = max(1, wins + losses + draws)
        rec = {"wins": wins, "losses": losses, "draws": draws, "win_rate": wins / total}
        results[opp] = rec
        if verbose:
            print(
                f"  vs {opp:<14} {wins}-{losses}-{draws}  "
                f"({100 * rec['win_rate']:.0f}%)",
                file=sys.stderr,
            )
    return results


## Source: cnn_v1 ppo

In [ ]:
"""Pure self-play PPO trainer for CNN v1.

No teacher, no behavior cloning. The learner plays against:
  - its current self (probability `self_play_ratio`), and
  - frozen snapshots of itself collected every `snapshot_every` iters.

Reward signal: potential-based shaping on planet-count + production
differential (`F = γΦ(s') − Φ(s)`), plus terminal ±1. Potential-based
shaping preserves optimal policy (Ng et al. 1999) so it cannot reward-hack.

Benchmark eval against `physical_v2` runs every `eval_every` iters and
gates the checkpoint — weights are only overwritten on the disk when
win rate improves over best-so-far.
"""

from __future__ import annotations

import copy
import json
import math
import random
import sys
import time
from collections import deque
from pathlib import Path
from typing import Callable

import torch
import torch.nn.functional as F
from torch.distributions import Bernoulli, Categorical, Normal

from kaggle_environments import make



MAX_PLANETS = 16
MAX_PROD = 50.0  # normalizer for production-sum differential (avg ~12 planets × ~3 prod)


def compute_potential(obs, learner_slot: int) -> float:
    """Φ(s) = (my_planets − enemy_planets)/16 + 0.5·(my_prod − enemy_prod)/MAX_PROD."""
    planets = obs.get("planets") if isinstance(obs, dict) else getattr(obs, "planets", None)
    if not planets:
        return 0.0
    my_p = enemy_p = 0
    my_prod = enemy_prod = 0
    for p in planets:
        _, owner, _, _, _, _, prod = p
        if owner == learner_slot:
            my_p += 1
            my_prod += prod
        elif owner >= 0:
            enemy_p += 1
            enemy_prod += prod
    return (my_p - enemy_p) / 16.0 + 0.5 * (my_prod - enemy_prod) / MAX_PROD


class StochasticPolicyAgent:
    """Wraps a CNNv1. Records a trajectory when `record=True`."""

    def __init__(self, model: CNNv1, training: bool = True, record: bool = True):
        self.model = model
        self.device = next(model.parameters()).device
        self.training = training
        self.record = record
        self.trajectory: list[dict] = []

    def reset_trajectory(self):
        self.trajectory = []

    def __call__(self, obs):
        return self._act(obs)

    def _act(self, obs):
        try:
            return self._act_impl(obs)
        except Exception as e:
            if not getattr(self, "_warned_exc", False):
                self._warned_exc = True
                print(f"[_act] EXCEPTION: {type(e).__name__}: {e}", flush=True)
            return []

    def _act_impl(self, obs):
        channels, scalars, my_planets = featurize(obs)
        moves = []
        if not my_planets:
            return moves

        channels = channels.to(self.device)
        scalars = scalars.to(self.device)

        with torch.no_grad():
            feat_map, scalar_feat = self.model(channels.unsqueeze(0), scalars.unsqueeze(0))
            coords = torch.tensor(
                [[x, y] for (_, x, y, _) in my_planets], dtype=torch.float32, device=self.device
            ).unsqueeze(0)
            launch, target, ship = _planet_features(
                self.model, channels.unsqueeze(0), scalars.unsqueeze(0), coords
            )
            value = self.model.value(feat_map, scalar_feat)
            frac_std = float(torch.exp(self.model.frac_log_std).item())

        launch, target, ship = launch.squeeze(0), target.squeeze(0), ship.squeeze(0)

        # Defensive: if the model produced NaN/Inf (diverged policy), emit no
        # moves rather than crashing the env with invalid actions. Training
        # will recover via the post-update NaN check; rollout just wastes the
        # episode.
        launch_ok = torch.isfinite(launch).all().item()
        target_ok = torch.isfinite(target).all().item()
        ship_ok = torch.isfinite(ship).all().item()
        value_ok = torch.isfinite(value).all().item()
        frac_ok = math.isfinite(frac_std) and frac_std > 0
        if not (launch_ok and target_ok and ship_ok and value_ok and frac_ok):
            # One-shot diagnostic per agent instance so we know which head is
            # producing garbage when this fires.
            if not getattr(self, "_warned_nan", False):
                self._warned_nan = True
                print(
                    f"[_act] non-finite output: launch_ok={launch_ok} "
                    f"target_ok={target_ok} ship_ok={ship_ok} "
                    f"value_ok={value_ok} frac_std={frac_std}",
                    flush=True,
                )
            return moves

        if self.training:
            launch_actions = Bernoulli(logits=launch).sample()
            target_actions = Categorical(logits=target).sample()
            frac_mean = torch.sigmoid(ship)
            frac_actions = Normal(frac_mean, frac_std).sample().clamp(0.01, 1.0)
        else:
            launch_actions = (torch.sigmoid(launch) > LAUNCH_THRESHOLD).float()
            target_actions = target.argmax(dim=-1)
            frac_actions = torch.sigmoid(ship)

        if self.record:
            lp = Bernoulli(logits=launch).log_prob(launch_actions).sum()
            tp = Categorical(logits=target).log_prob(target_actions) * launch_actions
            fp = Normal(torch.sigmoid(ship), frac_std).log_prob(frac_actions) * launch_actions
            log_prob = lp + tp.sum() + fp.sum()

            player = obs.get("player", 0) if isinstance(obs, dict) else getattr(obs, "player", 0)
            phi = compute_potential(obs, player)

            # Store on CPU so the buffer doesn't pin GPU memory.
            self.trajectory.append(
                {
                    "channels": channels.detach().cpu(),
                    "scalars": scalars.detach().cpu(),
                    "coords": coords.detach().squeeze(0).cpu(),
                    "launch_act": launch_actions.detach().cpu(),
                    "target_act": target_actions.detach().cpu(),
                    "frac_act": frac_actions.detach().cpu(),
                    "log_prob": float(log_prob.item()),
                    "value": float(value.item()),
                    "phi": float(phi),
                }
            )

        for i, (pid, x, y, ships) in enumerate(my_planets):
            if launch_actions[i].item() < 0.5:
                continue
            tc = int(target_actions[i].item())
            tcy, tcx = divmod(tc, GRID)
            tx = (tcx + 0.5) * CELL
            ty = (tcy + 0.5) * CELL
            n = max(1, min(int(ships), int(round(frac_actions[i].item() * ships))))
            if n < 1:
                continue
            angle = math.atan2(ty - y, tx - x)
            moves.append([pid, angle, n])
        return moves


def _wrap_as_fn(sp_agent: StochasticPolicyAgent) -> Callable:
    """Convert a StochasticPolicyAgent into a plain callable for env.run."""
    def fn(obs):
        return sp_agent(obs)
    return fn


_WARNED_SHORT_GAME = [False]


def _play_episode(
    learner: StochasticPolicyAgent,
    opponent_fn: Callable,
    learner_slot: int,
    seed: int | None,
):
    config = {"seed": seed} if seed is not None else {}
    env = make("orbit_wars", configuration=config, debug=False)
    learner.reset_trajectory()

    learner_fn = _wrap_as_fn(learner)
    players = [learner_fn, opponent_fn] if learner_slot == 0 else [opponent_fn, learner_fn]
    env.run(players)
    final_reward = env.steps[-1][learner_slot].reward or 0

    # Diagnostic: if the game ended much earlier than expected, surface the
    # per-player status + info so we can see WHY (TIMEOUT / INVALID / ERROR).
    if len(env.steps) < 50 and not _WARNED_SHORT_GAME[0]:
        _WARNED_SHORT_GAME[0] = True
        final = env.steps[-1]
        statuses = [getattr(s, "status", "?") for s in final]
        infos = [getattr(s, "info", None) for s in final]
        print(
            f"[short game] steps={len(env.steps)} learner_slot={learner_slot} "
            f"statuses={statuses} infos={infos}",
            flush=True,
        )
    return learner.trajectory, final_reward, env


def _compute_gae(rewards: list[float], values: list[float], gamma: float, lam: float):
    T = len(values)
    adv = [0.0] * T
    last = 0.0
    for t in reversed(range(T)):
        next_v = values[t + 1] if t + 1 < T else 0.0
        delta = rewards[t] + gamma * next_v - values[t]
        adv[t] = last = delta + gamma * lam * last
    returns = [a + v for a, v in zip(adv, values)]
    return adv, returns


def _pack_trajectory(
    traj: list[dict],
    final_reward: float,
    gamma: float,
    lam: float,
    use_shaping: bool,
):
    if not traj:
        return None
    T = len(traj)
    values = [s["value"] for s in traj]
    phis = [s["phi"] for s in traj]

    rewards = [0.0] * T
    if use_shaping:
        for t in range(T - 1):
            rewards[t] = gamma * phis[t + 1] - phis[t]
        rewards[T - 1] = final_reward - phis[T - 1]
    else:
        rewards[T - 1] = final_reward

    adv, ret = _compute_gae(rewards, values, gamma, lam)

    channels = torch.stack([s["channels"] for s in traj])
    scalars = torch.stack([s["scalars"] for s in traj])
    log_prob = torch.tensor([s["log_prob"] for s in traj])
    adv_t = torch.tensor(adv, dtype=torch.float32)
    ret_t = torch.tensor(ret, dtype=torch.float32)

    M = MAX_PLANETS
    coords = torch.zeros(T, M, 2)
    launch = torch.zeros(T, M)
    target = torch.zeros(T, M, dtype=torch.long)
    frac = torch.zeros(T, M)
    mask = torch.zeros(T, M)
    for i, s in enumerate(traj):
        n = min(M, s["coords"].size(0))
        coords[i, :n] = s["coords"][:n]
        launch[i, :n] = s["launch_act"][:n]
        target[i, :n] = s["target_act"][:n]
        frac[i, :n] = s["frac_act"][:n]
        mask[i, :n] = 1.0

    return {
        "channels": channels,
        "scalars": scalars,
        "coords": coords,
        "launch_act": launch,
        "target_act": target,
        "frac_act": frac,
        "mask": mask,
        "log_prob": log_prob,
        "adv": adv_t,
        "ret": ret_t,
    }


FRAC_LOG_STD_MIN = math.log(0.05)
FRAC_LOG_STD_MAX = math.log(1.0)


def _ppo_update(
    model: CNNv1,
    batch: dict,
    clip: float = 0.2,
    value_coef: float = 0.5,
    entropy_coef: float = 0.01,
    epochs: int = 2,
    minibatch: int = 32,
    lr: float = 1e-4,
    kl_stop: float = 0.4,
    ratio_max: float = 3.0,
    max_grad_norm: float = 0.1,
):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    device = next(model.parameters()).device
    N = batch["channels"].size(0)
    old_log_prob = batch["log_prob"].to(device)
    adv = batch["adv"].to(device)
    ret = batch["ret"].to(device)

    # Advantage normalization — standard PPO stabilizer. Skip if all-zero (degenerate).
    if adv.std() > 1e-6:
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    else:
        adv = adv - adv.mean()

    clip_frac_sum = 0.0
    approx_kl_sum = 0.0
    n_mb = skipped = 0
    stop_early = False
    last_policy = last_value = last_entropy = 0.0
    last_frac_std = float(torch.exp(model.frac_log_std).item())
    epoch_times: list[float] = []

    for _ in range(epochs):
        if stop_early:
            break
        epoch_t0 = time.time()
        idx = torch.randperm(N)
        for start in range(0, N, minibatch):
            sel = idx[start : start + minibatch]
            ch = batch["channels"][sel].to(device)
            sc = batch["scalars"][sel].to(device)
            co = batch["coords"][sel].to(device)
            la = batch["launch_act"][sel].to(device)
            ta = batch["target_act"][sel].to(device)
            fa = batch["frac_act"][sel].to(device)
            mk = batch["mask"][sel].to(device)

            feat_map = model.conv(ch)
            scalar_feat = model.scalar_mlp(sc)
            value = model.value(feat_map, scalar_feat)

            launch, target, ship = _planet_features(model, ch, sc, co)

            frac_std = torch.exp(model.frac_log_std)
            launch_lp = Bernoulli(logits=launch).log_prob(la) * mk
            target_lp = Categorical(logits=target).log_prob(ta) * la * mk
            ship_lp = Normal(torch.sigmoid(ship), frac_std).log_prob(fa) * la * mk
            new_log_prob = (launch_lp + target_lp + ship_lp).sum(dim=-1)

            # Per-step log_prob is a sum over ~16 planets; small weight
            # drift can blow up the exp. Clamp the ratio to keep the loss
            # scale sane while preserving the PPO clip surface.
            log_ratio = (new_log_prob - old_log_prob[sel]).clamp(
                min=math.log(1.0 / ratio_max), max=math.log(ratio_max)
            )
            ratio = torch.exp(log_ratio)
            unclipped = ratio * adv[sel]
            clipped = torch.clamp(ratio, 1 - clip, 1 + clip) * adv[sel]
            policy_loss = -torch.min(unclipped, clipped).mean()

            value_loss = F.mse_loss(value, ret[sel])

            launch_ent = Bernoulli(logits=launch).entropy() * mk
            target_ent = Categorical(logits=target).entropy() * mk
            entropy = (launch_ent + target_ent).sum(dim=-1).mean()

            loss = policy_loss + value_coef * value_loss - entropy_coef * entropy

            if not torch.isfinite(loss):
                skipped += 1
                continue

            opt.zero_grad()
            loss.backward()
            # NaN/Inf grad guard — zero them out instead of stepping into NaN space.
            bad_grad = False
            for p in model.parameters():
                if p.grad is not None and not torch.isfinite(p.grad).all():
                    bad_grad = True
                    break
            if bad_grad:
                skipped += 1
                continue
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
            opt.step()

            # Keep learned exploration std in a sane range.
            with torch.no_grad():
                model.frac_log_std.clamp_(FRAC_LOG_STD_MIN, FRAC_LOG_STD_MAX)

            clip_frac_sum += float(((ratio - 1.0).abs() > clip).float().mean().item())
            # Schulman's approx KL = E[(ratio - 1) - log(ratio)]; always ≥ 0.
            approx_kl = float(((ratio - 1.0) - log_ratio).mean().item())
            approx_kl_sum += approx_kl
            n_mb += 1
            last_policy = float(policy_loss.item())
            last_value = float(value_loss.item())
            last_entropy = float(entropy.item())
            last_frac_std = float(frac_std.item())

            if approx_kl > kl_stop:
                stop_early = True
                break

        epoch_times.append(time.time() - epoch_t0)

    return {
        "policy_loss": last_policy,
        "value_loss": last_value,
        "entropy": last_entropy,
        "clip_frac": clip_frac_sum / max(1, n_mb),
        "approx_kl": approx_kl_sum / max(1, n_mb),
        "frac_std": last_frac_std,
        "skipped_mb": skipped,
        "early_stopped": stop_early,
        "epoch_times": epoch_times,
    }


def _freeze(snap: CNNv1) -> CNNv1:
    snap.eval()
    for p in snap.parameters():
        p.requires_grad_(False)
    return snap


def _eval_winrate(
    model: CNNv1, opponent_id: str, games: int, seed_start: int
) -> float:
    """Write current weights so cnn_v1_agent sees them, then run eval."""
    WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), WEIGHTS_PATH)
    reload_weights()
    result = evaluate_agent(
        "cnn_v1", [opponent_id], games_per=games, seed_start=seed_start, verbose=False
    )
    reload_weights()
    return result[opponent_id]["win_rate"]


def _save_state_and_reload(state: dict) -> None:
    WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, WEIGHTS_PATH)
    reload_weights()


LATEST_PATH = WEIGHTS_PATH.parent / "latest.pt"


def _save_latest(model: CNNv1, iter_num: int) -> None:
    """Checkpoint current (not-best) weights to disk so training can resume.

    Plain state_dict (same format as cnn_v1.pt) so load_model() accepts it.
    """
    LATEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), LATEST_PATH)


def _has_nan_params(model: CNNv1) -> bool:
    for p in model.parameters():
        if not torch.isfinite(p).all():
            return True
    return False


def _fmt_duration(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.0f}s"
    if seconds < 3600:
        return f"{seconds / 60:.1f}m"
    return f"{seconds / 3600:.1f}h"


def train_ppo(
    iterations: int = 200,
    episodes_per_iter: int = 32,
    opponents: list[str] | None = None,
    snapshot_every: int = 10,
    snapshot_pool_size: int = 8,
    self_play_ratio: float = 0.3,
    eval_every: int = 10,
    eval_games: int = 20,
    eval_opponent: str = "physical_v2",
    resume_from: Path | str | None = None,
    save_to: Path | str | None = None,
    gamma: float = 0.99,
    lam: float = 0.95,
    use_shaping: bool = True,
    seed_start: int = 0,
    log_path: Path | str | None = None,
    device: str | None = None,
    episode_progress: bool = True,
    replay_every: int = 10,
    replay_dir: Path | str | None = None,
    checkpoint_dir: Path | str | None = None,
    kl_stop_start: float = 0.4,
    kl_stop_end: float | None = 0.05,
    verbose: bool = True,
) -> CNNv1:
    """Pure self-play PPO with a snapshot ring buffer.

    - `opponents`: external opponents (e.g. physics agents). Empty/None by
      default → pure self-play. Kept as a hook for ablations; sample with
      a small probability alongside snapshots if non-empty.
    - `snapshot_every`: iters between snapshots.
    - `self_play_ratio`: P(self) when snapshot pool non-empty; otherwise
      self is used 100%.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(device)

    # External checkpoint dir (e.g. mounted Google Drive, GCS) — survives
    # Colab runtime shutdown. Checked first for resume, written to every iter.
    ext_ckpt_dir = Path(checkpoint_dir) if checkpoint_dir else None
    if ext_ckpt_dir is not None:
        ext_ckpt_dir.mkdir(parents=True, exist_ok=True)
    ext_latest = ext_ckpt_dir / "latest.pt" if ext_ckpt_dir else None
    ext_best = ext_ckpt_dir / "cnn_v1.pt" if ext_ckpt_dir else None

    # Auto-resume: prefer external checkpoint (if provided and exists),
    # then local latest.pt, then local cnn_v1.pt, else fresh.
    resume_msg = "fresh weights"
    if resume_from is None:
        if ext_latest is not None and ext_latest.exists():
            resume_from = ext_latest
            resume_msg = f"resumed ext latest: {ext_latest}"
        elif ext_best is not None and ext_best.exists():
            resume_from = ext_best
            resume_msg = f"resumed ext best: {ext_best}"
        elif LATEST_PATH.exists():
            resume_from = LATEST_PATH
            resume_msg = f"resumed local latest: {LATEST_PATH}"
        elif WEIGHTS_PATH.exists():
            resume_from = WEIGHTS_PATH
            resume_msg = f"resumed local best: {WEIGHTS_PATH}"
    else:
        resume_msg = f"resumed: {resume_from}"

    model = load_model(resume_from) if resume_from else fresh_model()

    # GPU compat probe: torch 2.10+cu128 drops SM_60 (P100). If the assigned
    # GPU can't run torch kernels, fall back to CPU so training still runs.
    if device.type == "cuda":
        try:
            _probe = torch.zeros(4, device=device) + 1.0
            _probe.cpu()
        except Exception as e:
            print(
                f"WARN: GPU unusable ({type(e).__name__}: {e}); falling back to CPU. "
                f"On Kaggle: Settings → Accelerator → GPU T4 x2 to avoid this.",
                flush=True,
            )
            device = torch.device("cpu")

    model = model.to(device)

    # Emit device line BEFORE warmup so the log is never empty on a GPU error.
    if verbose:
        gpu_info = ""
        if device.type == "cuda":
            try:
                gpu_info = f" ({torch.cuda.get_device_name(device)})"
            except Exception:
                pass
        param_count = sum(p.numel() for p in model.parameters())
        kl_sched = (
            f"kl_stop: {kl_stop_start}→{kl_stop_end}"
            if kl_stop_end is not None
            else f"kl_stop: {kl_stop_start}"
        )
        print(
            f"device: {device}{gpu_info}  params: {param_count / 1e6:.2f}M  "
            f"iters: {iterations}  ep/iter: {episodes_per_iter}  {kl_sched}  "
            f"{resume_msg}",
            flush=True,
        )

    # Warmup: first CUDA forward pass triggers context init which can take
    # several seconds — well over orbit_wars' 1s actTimeout.
    try:
        with torch.no_grad():
            _warm_ch = torch.zeros(1, NUM_CHANNELS, GRID, GRID, device=device)
            _warm_sc = torch.zeros(1, SCALAR_DIM, device=device)
            _warm_coords = torch.zeros(1, 1, 2, device=device)
            _fm, _sf = model(_warm_ch, _warm_sc)
            _planet_features(model, _warm_ch, _warm_sc, _warm_coords)
            model.value(_fm, _sf)
            if device.type == "cuda":
                torch.cuda.synchronize(device)
    except Exception as e:
        print(f"WARN: warmup failed ({type(e).__name__}: {e}); continuing anyway", flush=True)

    learner = StochasticPolicyAgent(model, training=True, record=True)
    snapshots: deque[CNNv1] = deque(maxlen=snapshot_pool_size)

    ext_opponents = list(opponents or [])

    best_winrate = -1.0
    best_state = copy.deepcopy(model.state_dict())

    run_id = time.strftime("%Y%m%dT%H%M%S")
    if log_path is None:
        log_path = TRAIN_ROOT / f"ppo_{run_id}.jsonl"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_f = log_path.open("w")

    if replay_dir is None:
        replay_dir = TRAIN_ROOT.parent / "replays" / "training" / run_id
    replay_dir = Path(replay_dir)
    if replay_every > 0:
        replay_dir.mkdir(parents=True, exist_ok=True)

    train_start = time.time()
    try:
        for it in range(iterations):
            t0 = time.time()
            buffers: list[dict] = []
            rewards_hist: list[float] = []
            ep_types = {"self": 0, "snapshot": 0, "external": 0}
            ep_steps: list[int] = []

            last_env = None
            last_opp_kind = ""
            for ep in range(episodes_per_iter):
                r = random.random()
                if ext_opponents and r < 0.1:
                    opp_id = random.choice(ext_opponents)
                    opp_fn = Agent(id=opp_id).fn
                    ep_types["external"] += 1
                    opp_kind = f"ext:{opp_id}"
                elif not snapshots or r < 0.1 + self_play_ratio:
                    # Self-play against current model (stochastic, non-recording).
                    self_opp = StochasticPolicyAgent(model, training=True, record=False)
                    opp_fn = _wrap_as_fn(self_opp)
                    ep_types["self"] += 1
                    opp_kind = "self"
                else:
                    snap = random.choice(list(snapshots))
                    snap_agent = StochasticPolicyAgent(snap, training=True, record=False)
                    opp_fn = _wrap_as_fn(snap_agent)
                    ep_types["snapshot"] += 1
                    opp_kind = "snapshot"

                slot = ep % 2
                ep_t0 = time.time()
                traj, reward, env = _play_episode(
                    learner, opp_fn, learner_slot=slot, seed=seed_start + it * 1000 + ep
                )
                rewards_hist.append(reward)
                packed = _pack_trajectory(traj, reward, gamma, lam, use_shaping)
                if packed is not None:
                    buffers.append(packed)

                last_env = env
                last_opp_kind = opp_kind
                steps = len(env.steps)
                ep_steps.append(steps)

                if episode_progress:
                    wins_so_far = sum(1 for r in rewards_hist if r > 0.5)
                    losses_so_far = sum(1 for r in rewards_hist if r < -0.5)
                    draws_so_far = len(rewards_hist) - wins_so_far - losses_so_far
                    iter_elapsed = time.time() - t0
                    print(
                        f"  iter {it + 1:3d} ep {ep + 1:2d}/{episodes_per_iter}  "
                        f"opp={opp_kind:12s}  slot={slot}  r={reward:+.0f}  "
                        f"steps={steps:3d}  "
                        f"W/L/D={wins_so_far}/{losses_so_far}/{draws_so_far}  "
                        f"ep={time.time() - ep_t0:.1f}s  "
                        f"iter={_fmt_duration(iter_elapsed)}",
                        flush=True,
                    )

            if not buffers:
                # All rollouts returned empty trajectories — usually policy
                # collapse. Print a loud notice instead of silently skipping,
                # so the user sees iter-level signal even in this failure mode.
                wins = sum(1 for r in rewards_hist if r > 0.5)
                losses = sum(1 for r in rewards_hist if r < -0.5)
                draws = len(rewards_hist) - wins - losses
                total_elapsed = time.time() - train_start
                print(
                    f"iter {it + 1:3d}/{iterations}  EMPTY_BUFFERS  "
                    f"W/L/D={wins}/{losses}/{draws}  "
                    f"opp={ep_types['self']}s/{ep_types['snapshot']}p/{ep_types['external']}x  "
                    f"elapsed={_fmt_duration(total_elapsed)}  "
                    f"(policy likely diverged; reverting to best_state)",
                    flush=True,
                )
                # Revert to the last-known-good state and continue.
                model.load_state_dict(best_state)
                _save_latest(model, it + 1)
                continue

            batch = {k: torch.cat([b[k] for b in buffers], dim=0) for k in buffers[0]}

            # Dynamic kl_stop: linear decay from kl_stop_start to kl_stop_end
            # as training progresses (tighter tolerance late, more exploration early).
            if kl_stop_end is not None and iterations > 1:
                progress = it / (iterations - 1)
                kl_stop_now = kl_stop_start + (kl_stop_end - kl_stop_start) * progress
            else:
                kl_stop_now = kl_stop_start
            stats = _ppo_update(model, batch, kl_stop=kl_stop_now)

            # NaN recovery: if the update produced non-finite params, revert
            # to best_state and keep training from there instead of emitting
            # garbage for the rest of the run.
            if _has_nan_params(model):
                if verbose:
                    print(
                        f"  iter {it + 1}: NaN in params detected; reverting to best_state",
                        flush=True,
                    )
                model.load_state_dict(best_state)

            if (it + 1) % snapshot_every == 0:
                snap = _freeze(copy.deepcopy(model))
                snapshots.append(snap)

            # Checkpoint the current model every iter so training can resume
            # from the most recent state (separate from best-by-eval cnn_v1.pt).
            _save_latest(model, it + 1)
            if ext_latest is not None:
                try:
                    torch.save(model.state_dict(), ext_latest)
                except Exception as e:
                    if verbose:
                        print(f"  ext latest save failed: {e}", flush=True)

            # Flush a replay of the last episode periodically.
            if replay_every > 0 and (it + 1) % replay_every == 0 and last_env is not None:
                rpath = replay_dir / f"iter_{it + 1:03d}_{last_opp_kind.replace(':', '_')}.html"
                try:
                    rpath.write_text(last_env.render(mode="html"))
                    if verbose:
                        print(f"  replay: {rpath}", flush=True)
                except Exception as e:
                    if verbose:
                        print(f"  replay save failed: {e}", flush=True)

            eval_wr = None
            if (it + 1) % eval_every == 0:
                eval_wr = _eval_winrate(
                    model, eval_opponent, eval_games, seed_start=100_000 + it
                )
                if eval_wr > best_winrate:
                    best_winrate = eval_wr
                    best_state = copy.deepcopy(model.state_dict())
                    # current weights already saved on disk by _eval_winrate;
                    # also mirror best to external dir.
                    if ext_best is not None:
                        try:
                            torch.save(best_state, ext_best)
                        except Exception as e:
                            if verbose:
                                print(f"  ext best save failed: {e}", flush=True)
                else:
                    _save_state_and_reload(best_state)

            mean_r = sum(rewards_hist) / max(1, len(rewards_hist))
            wins = sum(1 for r in rewards_hist if r > 0.5)
            losses = sum(1 for r in rewards_hist if r < -0.5)
            draws = len(rewards_hist) - wins - losses
            avg_steps = sum(ep_steps) / max(1, len(ep_steps))
            iter_time = time.time() - t0
            total_elapsed = time.time() - train_start
            iters_done = it + 1
            iters_remaining = iterations - iters_done
            eta_s = (total_elapsed / iters_done) * iters_remaining if iters_done else 0
            gpu_mem_gb = None
            if device.type == "cuda":
                try:
                    gpu_mem_gb = torch.cuda.max_memory_allocated(device) / 1e9
                except Exception:
                    pass

            row = {
                "iter": it + 1,
                "mean_reward": round(mean_r, 4),
                "wins": wins,
                "losses": losses,
                "draws": draws,
                "avg_episode_steps": round(avg_steps, 1),
                "policy_loss": round(stats["policy_loss"], 4),
                "value_loss": round(stats["value_loss"], 4),
                "entropy": round(stats["entropy"], 4),
                "clip_frac": round(stats["clip_frac"], 4),
                "approx_kl": round(stats["approx_kl"], 4),
                "frac_std": round(stats["frac_std"], 4),
                "samples": int(batch["channels"].size(0)),
                "snapshot_count": len(snapshots),
                "ep_types": ep_types,
                "early_stopped": bool(stats["early_stopped"]),
                "skipped_mb": int(stats["skipped_mb"]),
                "epoch_times_s": [round(t, 3) for t in stats.get("epoch_times", [])],
                "kl_stop_now": round(kl_stop_now, 4),
                "time_s": round(iter_time, 2),
                "elapsed_s": round(total_elapsed, 2),
                "eta_s": round(eta_s, 2),
            }
            if gpu_mem_gb is not None:
                row["gpu_mem_gb"] = round(gpu_mem_gb, 3)
            if eval_wr is not None:
                row["eval_winrate"] = round(eval_wr, 3)
                row["best_winrate"] = round(best_winrate, 3)

            log_f.write(json.dumps(row) + "\n")
            log_f.flush()

            if verbose:
                eval_str = (
                    f"  eval={eval_wr:.2f}(best={best_winrate:.2f})"
                    if eval_wr is not None
                    else ""
                )
                es = "*" if stats["early_stopped"] else " "
                mem_str = f"  mem={gpu_mem_gb:.1f}G" if gpu_mem_gb is not None else ""
                ep_types_str = (
                    f"{ep_types['self']}s/{ep_types['snapshot']}p/{ep_types['external']}x"
                )
                epoch_t = stats.get("epoch_times", [])
                epoch_str = "/".join(f"{t:.1f}" for t in epoch_t) if epoch_t else "-"
                print(
                    f"iter {it + 1:3d}/{iterations}  "
                    f"W/L/D={wins}/{losses}/{draws}  "
                    f"r={mean_r:+.3f}  steps={avg_steps:.0f}  "
                    f"pi={stats['policy_loss']:+.2f}  v={stats['value_loss']:.3f}  "
                    f"ent={stats['entropy']:.1f}  kl={stats['approx_kl']:.3f}{es} "
                    f"clip={stats['clip_frac']:.2f}  std={stats['frac_std']:.2f}  "
                    f"opp={ep_types_str}  snap={len(snapshots)}  "
                    f"n={batch['channels'].size(0)}  "
                    f"ep_t={epoch_str}s  "
                    f"t={_fmt_duration(iter_time)} "
                    f"elapsed={_fmt_duration(total_elapsed)} "
                    f"ETA={_fmt_duration(eta_s)}"
                    f"{mem_str}{eval_str}",
                    flush=True,
                )
    finally:
        log_f.close()

    model.load_state_dict(best_state)
    model.eval()
    path = save_model(model, save_to)
    reload_weights()
    if verbose:
        print(
            f"saved best weights: {path} (best_winrate={best_winrate:.3f}, "
            f"log: {log_path})",
            flush=True,
        )
    return model


## Verify

In [ ]:
print("registered agents:", list_agents())


## Train

In [ ]:
import time
eps_per_iter = 32 if (IS_KAGGLE or IS_COLAB) else 16
t0 = time.time()
train_ppo(
    iterations=100, episodes_per_iter=eps_per_iter,
    snapshot_every=10, snapshot_pool_size=8, self_play_ratio=0.3,
    eval_every=5, eval_games=20, eval_opponent='physical_v2',
    use_shaping=True, episode_progress=True, replay_every=5,
    checkpoint_dir=CKPT_DIR, kl_stop_start=0.4, kl_stop_end=0.05,
    device='cuda', verbose=True,
)
print(f'total: {time.time() - t0:.0f}s')


## Save logs + replays

In [ ]:
if IS_COLAB: dst = Path('/content/drive/MyDrive/orbit-wars')
elif IS_KAGGLE: dst = Path('/kaggle/working')
else: dst = Path('./out').resolve()
dst.mkdir(parents=True, exist_ok=True)
for jsonl in sorted(Path('logs/training').glob('ppo_*.jsonl')):
    shutil.copy(jsonl, dst / jsonl.name)
replays_src = Path('logs/replays/training')
if replays_src.exists():
    for run_dir in replays_src.iterdir():
        out = dst / 'replays' / run_dir.name
        out.mkdir(parents=True, exist_ok=True)
        for html in run_dir.glob('*.html'):
            shutil.copy(html, out / html.name)
print('copied to', dst)
print('checkpoints at:', list(CKPT_DIR.iterdir()))
